# Forecasting de ventas 2025

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import jupyter
import streamlit as st
import holidays

# Carga de archivo de ventas para inferencia

<small>Se cargan las ventas desde el archivo de inferencia y se interpreta `fecha` como dato temporal.</small>

In [18]:
ruta_inferencia = '../data/raw/inferencia/ventas_2025_inferencia.csv'
inferencia_df = pd.read_csv(ruta_inferencia, parse_dates=['fecha'])

# Mostrar las primeras filas para verificar la carga
display(inferencia_df.head())

,fecha,producto_id,nombre,categoria,subcategoria,precio_base,es_estrella,unidades_vendidas,precio_venta,ingresos,Amazon,Decathlon,Deporvillage
0,2025-10-25,PROD_001,Nike Air Zoom Pegasus 40,Running,Zapatillas Running,115,True,26.0,113.13,2941.38,89.51,113.43,104.78
1,2025-10-25,PROD_002,Adidas Ultraboost 23,Running,Zapatillas Running,135,True,27.0,141.89,3831.03,128.73,112.91,122.88
2,2025-10-25,PROD_003,Asics Gel Nimbus 25,Running,Zapatillas Running,85,False,5.0,85.79,428.95,84.28,74.51,85.57
3,2025-10-25,PROD_004,New Balance Fresh Foam X 1080v12,Running,Zapatillas Running,75,False,3.0,76.19,228.57,75.54,70.32,71.13
4,2025-10-25,PROD_005,Nike Dri-FIT Miler,Running,Ropa Running,35,False,3.0,35.48,106.44,33.84,31.32,34.41


## Transformación de datos para inferencia

Se replica el pipeline de entrenamiento (variables temporales, festivos, lags, medias móviles, precios y One-Hot Encoding) sobre `inferencia_df`. Los lags y medias móviles se calculan antes de filtrar noviembre para conservar la ventana de octubre. Finalmente se alinean las columnas con `X_train`, se guarda el CSV y se mantienen los nulos esperados en los lags de noviembre.

In [19]:
# --- Reconstruir X_train si no está en memoria ---
try:
    X_train.columns
except NameError:
    df_entrenamiento = pd.read_csv('../data/processed/df.csv', parse_dates=['fecha'])
    columnas_excluidas = ['unidades_vendidas', 'fecha', 'ingresos']
    X_train = df_entrenamiento.drop(columns=columnas_excluidas, errors='ignore').select_dtypes(include=['number', 'bool'])

df = inferencia_df.copy()

# 1. Fechas
fechas_df = pd.to_datetime(df['fecha'])
df['fecha'] = fechas_df
anios_df = sorted(fechas_df.dt.year.dropna().unique().tolist())
calendario_festivos_es = holidays.country_holidays('ES', years=[int(a) for a in anios_df])

df['anio'] = fechas_df.dt.year
df['mes'] = fechas_df.dt.month
df['dia_mes'] = fechas_df.dt.day
df['dia_semana'] = fechas_df.dt.dayofweek
df['nombre_dia_semana'] = fechas_df.dt.dayofweek.map(
    dict(enumerate(['Lunes','Martes','Miercoles','Jueves','Viernes','Sabado','Domingo'])))
df['trimestre'] = fechas_df.dt.quarter
df['dia_del_anio'] = fechas_df.dt.dayofyear
df['semana_iso'] = fechas_df.dt.isocalendar().week.astype('int64')
df['es_fin_de_semana'] = df['dia_semana'].isin([5, 6])
df['es_inicio_mes'] = df['dia_mes'] <= 3
df['es_fin_mes'] = df['dia_mes'] > (fechas_df.dt.days_in_month - 3)
df['mes_sin'] = np.sin(2*np.pi*(df['mes']-1)/12)
df['mes_cos'] = np.cos(2*np.pi*(df['mes']-1)/12)
df['dia_semana_sin'] = np.sin(2*np.pi*df['dia_semana']/7)
df['dia_semana_cos'] = np.cos(2*np.pi*df['dia_semana']/7)

fechas_solo_dia = fechas_df.dt.date
df['es_festivo_es'] = fechas_solo_dia.isin(calendario_festivos_es)
df['nombre_festivo_es'] = fechas_solo_dia.map(calendario_festivos_es.get).fillna('No festivo')

fechas_black_friday = {}
for anio in anios_df:
    viernes_noviembre = pd.date_range(start=f'{int(anio)}-11-01', end=f'{int(anio)}-11-30', freq='W-FRI')
    fechas_black_friday[int(anio)] = viernes_noviembre[-1]
fechas_cyber_monday = {a: f + pd.Timedelta(days=3) for a, f in fechas_black_friday.items()}

fechas_normalizadas = fechas_df.dt.normalize()
df['es_black_friday'] = fechas_normalizadas.isin(fechas_black_friday.values())
df['es_cyber_monday'] = fechas_normalizadas.isin(fechas_cyber_monday.values())
fechas_campana_bf = [f + pd.Timedelta(days=d) for f in fechas_black_friday.values() for d in range(4)]
df['es_campana_black_friday'] = fechas_normalizadas.isin(fechas_campana_bf)

# 2. Lags y medias móviles (antes de filtrar)
df = df.sort_values(['producto_id', 'anio', 'fecha']).reset_index(drop=True)
grupo_unidades = df.groupby(['producto_id', 'anio'], sort=False)['unidades_vendidas']
for periodo in range(1, 8):
    df[f'unidades_lag_{periodo}'] = grupo_unidades.shift(periodo)
df['media_movil_7_dias'] = grupo_unidades.transform(
    lambda s: s.shift(1).rolling(7, min_periods=7).mean())

# 3. Precios y competencia
df['descuento_porcentaje'] = (df['precio_venta'] - df['precio_base']) / df['precio_base'] * 100
competidores = ['Amazon', 'Decathlon', 'Deporvillage']
df['precio_competencia'] = df[competidores].mean(axis=1)
df['ratio_precio'] = df['precio_venta'] / df['precio_competencia']
df = df.drop(columns=competidores)

# 4. One-Hot Encoding
variables_categoricas_h = ['nombre_h', 'categoria_h', 'subcategoria_h']
variables_originales = ['nombre', 'categoria', 'subcategoria']
for vh, vo in zip(variables_categoricas_h, variables_originales):
    df[vh] = df[vo].copy()

columnas_one_hot = pd.get_dummies(df[variables_categoricas_h], prefix=variables_categoricas_h, dtype=int)
prefijos = tuple(f'{v}_' for v in variables_categoricas_h)
cols_existentes = [c for c in df.columns if c.startswith(prefijos)]
df = df.drop(columns=cols_existentes)
df = pd.concat([df, columnas_one_hot], axis=1)

print("Transformaciones completadas. df listo para filtrar.")

Transformaciones completadas. df listo para filtrar.


C:\Users\Vehiller\AppData\Local\Temp\ipykernel_6896\1705315643.py:20: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  df['dia_semana'] = fechas_df.dt.dayofweek
C:\Users\Vehiller\AppData\Local\Temp\ipykernel_6896\1705315643.py:21: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  df['nombre_dia_semana'] = fechas_df.dt.dayofweek.map(
C:\Users\Vehiller\AppData\Local\Temp\ipykernel_6896\1705315643.py:24: Pandas4Warning: Series.dt.dayofyear is deprecated and will be removed in a future version. Use Series.dt.day_of_year instead.
  df['dia_del_anio'] = fechas_df.dt.dayofyear


## Filtrado y guardado final

Filtrar solo noviembre y guardar el DataFrame transformado.

In [20]:
from pathlib import Path

# Filtrar solo noviembre (eliminar octubre)
# Los registros de octubre que tengan nulos en lags se eliminan aquí
# Los registros de noviembre se mantienen TODOS, aunque tengan nulos
print(f"Registros totales antes del filtrado: {len(df)}")
print(f"Registros de octubre: {len(df[df['mes'] == 10])}")
print(f"Registros de noviembre: {len(df[df['mes'] == 11])}")

# Eliminar todos los registros de octubre
df = df[df['mes'] == 11].copy()

print(f"\nRegistros después de filtrar solo noviembre: {len(df)}")
print(f"Fechas en inferencia_df: min={df['fecha'].min()}, max={df['fecha'].max()}")
print(f"Productos únicos: {df['producto_id'].nunique()}")

# Verificar si hay nulos en las variables de lags en noviembre
cols_lag = [f'unidades_lag_{i}' for i in range(1, 8)] + ['media_movil_7_dias']
nulos_nov = df[cols_lag].isnull().sum()
if nulos_nov.sum() > 0:
    print(f"\n⚠️ Atención: Hay {nulos_nov.sum()} valores nulos en las variables de lag en noviembre.")
    print(nulos_nov[nulos_nov > 0])
else:
    print("\n✅ No hay valores nulos en las variables de lag para noviembre")

# Eliminar columnas como en entrenamiento
df = df.drop(columns=['fecha', 'unidades_vendidas', 'ingresos'], errors='ignore')

# Alinear con X_train
columnas_faltantes = [c for c in X_train.columns if c not in df.columns]
columnas_ohe_faltantes = [c for c in columnas_faltantes if c.startswith(prefijos)]
otras_faltantes = [c for c in columnas_faltantes if c not in columnas_ohe_faltantes]
assert not otras_faltantes, f"Faltan columnas predictoras: {otras_faltantes}"

for c in columnas_ohe_faltantes:
    df[c] = 0

df = df[X_train.columns]
assert list(df.columns) == list(X_train.columns), "Las columnas no coinciden con X_train"

# Guardar
ruta_salida = Path('../data/processed/inferencia_df_trasnformado.csv')
ruta_salida.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(ruta_salida, index=False)

inferencia_df_trasnformado = df
print(f"\nGuardado en {ruta_salida.resolve()} | shape: {df.shape}")

Registros totales antes del filtrado: 888
Registros de octubre: 168
Registros de noviembre: 720

Registros después de filtrar solo noviembre: 720
Fechas en inferencia_df: min=2025-11-01 00:00:00, max=2025-11-30 00:00:00
Productos únicos: 24

⚠️ Atención: Hay 5064 valores nulos en las variables de lag en noviembre.
unidades_lag_1        696
unidades_lag_2        672
unidades_lag_3        648
unidades_lag_4        624
unidades_lag_5        600
unidades_lag_6        576
unidades_lag_7        552
media_movil_7_dias    696
dtype: int64

Guardado en C:\Users\Vehiller\Documents\GitHub\ds4b-experienc-ia-en-data-science\data\processed\inferencia_df_trasnformado.csv | shape: (720, 76)


In [23]:
# 1. Imprimir el resumen visual (usando el dataframe original que SÍ tiene las columnas)
print("="*50)
print("RESUMEN FINAL DEL DATAFRAME DE INFERENCIA")
print("="*50)

# Filtramos el original solo para noviembre para el resumen
resumen_nov = inferencia_df[inferencia_df['fecha'].dt.month == 11]

print(f"Total de registros: {len(resumen_nov)}")
print(f"Productos únicos: {resumen_nov['producto_id'].nunique()}")
print(f"Rango de fechas: {resumen_nov['fecha'].min()} a {resumen_nov['fecha'].max()}")
print("="*50)

# 2. Guardar el dataframe transformado (el que ya tiene las 76 columnas listas para el modelo)
# Usamos el 'df' que quedó en memoria de la celda anterior
df.to_csv('../data/processed/inferencia_df_trasnformado.csv', index=False)

print("\nDataFrame transformado guardado en data/processed/inferencia_df_trasnformado.csv")
print(f"Shape del archivo guardado: {df.shape}")

RESUMEN FINAL DEL DATAFRAME DE INFERENCIA
Total de registros: 720
Productos únicos: 24
Rango de fechas: 2025-11-01 00:00:00 a 2025-11-30 00:00:00

DataFrame transformado guardado en data/processed/inferencia_df_trasnformado.csv
Shape del archivo guardado: (720, 76)
